## Assignment 1
Two lakes A and B will be connected by a pipe, and from lake B water will be pumped to reservoir C.
The pipe length from A to B is 150 m, and between B and C is 200 m. Concrete pipes are utilized throughout
(wall roughness ks=0.002 m). The required discharge between A and B is Q=0.045 m3/s, and the required
discharge to be pumped to lake C is Q=0.085 m3/s. The diameter of pipe BC is fixed at D=0.2 m.
The following minor loss coefficients can be utilized: inlet=0.5 at pipe inlets, outlet=1.0 at pipe outlets, and
elbow=0.5 for the two right-angle kinks between lake B and C. Finally, recall that for a filled circular pipe the
hydraulic radius is equivalent to R=A/P=D/4

![alt text](image.png)

In [53]:
import numpy as np
from scipy.optimize import fsolve

In [54]:
# --------------------------
# Constants and known values
# --------------------------
Q_ab = 0.045            # Flow rate in pipe AB [m^3/s]
Q_bc = 0.085            # Flow rate in pipe BC [m^3/s]
l_ab = 150              # Pipe length [m]
l_bc = 200              # Pipe length [m]
d_bc = 0.2              # Diameter of pipe BC [m]
k_s = 0.002             # Wall roughness [m]
z_1 = 32.56             # Elevation at point A [m]
z_2 = 26.67             # Elevation at point B [m]
z_3 = 90.00             # Elevation at point C [m]
zeta_inlet = 0.5        # Local loss coefficient at inlet
zeta_outlet = 1.0       # Local loss coefficient at outlet
zeta_elbow = 0.5        # Local loss coefficient at elbow
g = 9.81                # Gravitational acceleration [m/s^2]
viscosity = 1.003e-6    # Kinematic viscosity [m^2/s]

### Question 1
Determine the pipe diameter D that will yield the required discharge from lake A to B, taking into account
head losses (i.e. both “minor losses” and “friction losses”). [HINT 1: Solve simultaneously the energy
equation and Colebrook-White equation for the two unknowns D and the friction coefficient f. HINT 2:
Use Maple’s fsolve or Mathematica’s FindRoot function.]

In [55]:
# --------------------------
# System of equations for AB
# --------------------------
def equations_AB(unknowns):
    f_ab, d_ab = unknowns

    # Cross-sectional area and velocity
    R_ab = d_ab / 4
    A_ab = np.pi / 4 * d_ab **2
    V_ab = Q_ab / A_ab
    Re_ab = R_ab * V_ab / viscosity

    # Colebrook equation
    colebrook = np.sqrt(2 / f_ab) - (6.4 - 2.45 * np.log(k_s / R_ab + 4.7 / (Re_ab * np.sqrt(f_ab))))

    # Energy equation
    H_loss_ab = (zeta_inlet + zeta_outlet) * V_ab**2 / (2 * g) + f_ab * l_ab / R_ab * V_ab**2 / (2 * g)
    energy_eq = z_1 - (z_2 + H_loss_ab)

    return [colebrook, energy_eq]


# --------------------------
# Solve the system AB
# --------------------------
initial_guess = [0.01, 0.1]  # [f_ab, d_ab]
solution = fsolve(equations_AB, initial_guess)
f_ab_sol, d_ab_sol = solution

In [56]:
# --------------------------
# Output results
# --------------------------
print(f"Friction factor f_ab: {f_ab_sol:.6f}")
print(f"Diameter d_ab: {d_ab_sol:.6f} m")

Friction factor f_ab: 0.010236
Diameter d_ab: 0.178669 m


### Question 2
Taking energy losses into account, how many meters head must the pump supply to deliver the desired
discharge from lake B to C? [HINT: Similarly, solve the energy equation and Colebrook-White equation
for the two unknowns Hpump and f.]

In [57]:
# --------------------------
# System of equations for BC
# --------------------------
def equations_BC(unknowns):
    f_bc, H_pump = unknowns

    # Cross-sectional area and velocity
    R_bc = d_bc / 4
    A_bc = np.pi / 4 * d_bc **2
    V_bc = Q_bc / A_bc
    Re_bc = R_bc * V_bc / viscosity

    # Colebrook equation
    colebrook = np.sqrt(2 / f_bc) - (6.4 - 2.45 * np.log(k_s / R_bc + 4.7 / (Re_bc * np.sqrt(f_bc))))

    # Energy equation
    H_loss_bc = (zeta_inlet + zeta_outlet + 2 * zeta_elbow) * V_bc**2 / (2 * g) + f_bc * l_bc / R_bc * V_bc**2 / (2 * g)
    energy_eq = z_2 + H_pump - (z_3 + H_loss_bc)

    return [colebrook, energy_eq]


# --------------------------
# Solve the system BC
# --------------------------
initial_guess_BC = [0.01, 50]  # [f_bc, H_pump]
solution = fsolve(equations_BC, initial_guess_BC)
f_bc_sol, H_pump = solution

In [58]:
# --------------------------
# Output results
# --------------------------
print(f"Friction factor f_bc: {f_bc_sol:.6f}")
print(f"Required pump head H_pump: {H_pump:.6f} m")

Friction factor f_bc: 0.009829
Required pump head H_pump: 78.931662 m


## Results sum up

In [59]:
print(f"Friction factor f_ab: {f_ab_sol:.6f}")
print(f"Friction factor f_bc: {f_bc_sol:.6f}")
print(f"Diameter d_ab: {d_ab_sol:.6f} m")
print(f"Required pump head H_pump: {H_pump:.2f} m")

Friction factor f_ab: 0.010236
Friction factor f_bc: 0.009829
Diameter d_ab: 0.178669 m
Required pump head H_pump: 78.93 m
